# 🌐 ns-3 in the Browser — Network Simulation with Python

**Summer Camp 2026 · Tuesday, June 16 · ns-3 Installation & Examples**

Yesterday you learned **Python**. This morning you saw **Ubuntu** and how researchers install ns-3.
Now we put it together — and the best part: **you don't have to install anything.**

> ### 💡 The big idea: *Colab is Ubuntu*
> Every Google Colab notebook runs on a real **Ubuntu (Linux)** machine in Google's cloud.
> When you put a `!` in front of a command, it runs the *exact* Ubuntu commands you saw on the projector.
> So `!pip install ns3` does all the hard installation work for you — on Google's computer, not yours.

**Today you will:** install ns-3 with one line → build networks with a few lines of Python → and *see* them drawn on screen. 🎨

## Step 1 — Install ns-3 (run this first!) ⏳

Click the cell below and press **▶︎** (or **Shift + Enter**). This takes about a minute.

The `!` means *"run this as an Ubuntu command"* — the same `pip install` idea you'd type in a real terminal.

In [ ]:
!pip install ns3

## Step 2 — Bring ns-3 into Python

Now we tell Python where ns-3 lives and import it. After this runs with no error, ns-3 is ready. ✅

In [ ]:
import sys
sys.path.append("./ns-3-dev/build/bindings/python")
sys.path.append("./ns-3-dev/build/lib")
from ns import ns

from matplotlib import pyplot as plt
print("ns-3 is ready! 🎉")

## Step 3 — A helper to *draw* our network 🎨

A network is made of **nodes** (think: computers, phones, routers).
Run the cell below once — it teaches the notebook how to draw nodes as dots.
*(You don't need to understand every line. Just run it.)*

In [ ]:
# Draw each node in the network as a dot on a map
def plotNodePosition(nodeContainer, showLegend=False):
    for node_i in range(nodeContainer.GetN()):
        node = nodeContainer.Get(node_i).__deref__()
        mobility = node.GetObject[ns.MobilityModel]().__deref__()
        position = mobility.GetPosition()
        plt.scatter(position.x, position.y, label=f"Node {node.GetId()}")
    plt.ylabel("Y (m)")
    plt.xlabel("X (m)")
    if showLegend:
        plt.legend()
    plt.show()

## Step 4 — Your first network: a line of nodes 📏

The cell below creates **12 nodes** and places them in rows. Run it and watch the dots appear!

Read the comments to see what each line does.

In [ ]:
def plotNodesInLineTopology(num_nodes=12):
    ns.Simulator.Destroy()

    nodes = ns.NodeContainer()
    nodes.Create(num_nodes)            # <-- create the network nodes

    positions = ns.CreateObject[ns.ListPositionAllocator]()
    placed = 0
    for line in range(3):              # 3 rows
        for column in range(6):        # 6 per row
            if placed < num_nodes:
                positions.__deref__().Add(ns.Vector(100*column, 100*line, 0))
                placed += 1

    mobility = ns.MobilityHelper()
    mobility.SetMobilityModel("ns3::ConstantPositionMobilityModel")
    mobility.SetPositionAllocator(positions)
    mobility.Install(nodes)

    plotNodePosition(nodes)

plotNodesInLineTopology(12)

### ✍️ Your Turn (2 min)

Change the `12` in the last line to a different number — try **your age**, then try **3**, then try **18**.
Re-run the cell each time (Shift + Enter).

**🔮 Predict first:** before you run it, how many dots do you think you'll see?

## Step 5 — Show the *connections*, not just the nodes 🔗

Real networks have **links** between nodes. This helper draws lines for the connections too.
Run it once (again, just run — don't worry about every detail).

In [ ]:
def plotNodePositionAndChannels(nodeContainer, showLegend=False):
    nodeIdToCoordinate = {}
    channelIdToNodeIds = {}
    for node_i in range(nodeContainer.GetN()):
        node = nodeContainer.Get(node_i).__deref__()
        nodeId = node.GetId()
        mobility = node.GetObject[ns.MobilityModel]().__deref__()
        position = mobility.GetPosition()
        nodeIdToCoordinate[nodeId] = (position.x, position.y)
        for device_i in range(node.GetNDevices()):
            netdevice = node.GetDevice(device_i).__deref__()
            if hasattr(netdevice, "GetChannel"):
                channel = netdevice.GetChannel().__deref__()
                channelId = channel.GetId()
                channelIdToNodeIds.setdefault(channelId, []).append(nodeId)
        plt.scatter(position.x, position.y, label=f"Node {nodeId}")

    for channelId, channelNodes in channelIdToNodeIds.items():
        for i in range(len(channelNodes)):
            for j in range(i, len(channelNodes)):
                a = nodeIdToCoordinate[channelNodes[i]]
                b = nodeIdToCoordinate[channelNodes[j]]
                plt.plot([a[0], b[0]], [a[1], b[1]], "gray", zorder=-1)

    plt.ylabel("Y (m)")
    plt.xlabel("X (m)")
    if showLegend:
        plt.legend()
    plt.show()

## Step 6 — A **star** network ⭐

One central **hub** connected to many **spokes** — like a Wi-Fi router with devices around it.

In [ ]:
def plotNodesInStarTopology(spokes=10):
    ns.Simulator.Destroy()

    pointToPoint = ns.PointToPointHelper()
    star = ns.PointToPointStarHelper(spokes, pointToPoint)
    star.BoundingBox(1, 1, 100, 100)

    nodes = ns.NodeContainer()
    nodes.Add(star.GetHub())
    for node_i in range(star.SpokeCount()):
        nodes.Add(star.GetSpokeNode(node_i))

    plotNodePositionAndChannels(nodes)

plotNodesInStarTopology(10)

## Step 7 — A **grid** network 🔲

Nodes arranged in a mesh — like cell towers covering a city, or sensors in a field.

In [ ]:
def plotNodesInGridTopology(rows=10, cols=7):
    ns.Simulator.Destroy()

    pointToPoint = ns.PointToPointHelper()
    grid = ns.PointToPointGridHelper(rows, cols, pointToPoint)
    grid.BoundingBox(1, 1, 100, 100)

    nodes = ns.NodeContainer()
    for i in range(rows):
        for j in range(cols):
            nodes.Add(grid.GetNode(i, j))

    plotNodePositionAndChannels(nodes)

plotNodesInGridTopology(10, 7)

### ✍️ Challenge — Pick One! (5 min) 🏆

Edit the numbers and re-run:

1. **Easy:** Make a star with **5** spokes. Then **25**. What happens to the picture?
2. **Medium:** Make a **square** grid (same rows and cols, e.g. `6, 6`).
3. **Spicy:** Make a *tall, skinny* grid — many rows, few columns (e.g. `15, 2`). What real network does it remind you of?

🗣️ **Share:** Show your neighbor your favorite network shape and say what it could be in real life (Wi-Fi? city sensors? the internet?).

## 🎬 Bonus — Moving nodes (instructor demo)

ns-3 can also **animate** nodes that move over time (think: a phone in a moving car, or a drone).
That uses some advanced C++/Python interop, so just **watch** this one — you'll build toward it later in the week!

---

## 🎉 What you just did

In about an hour, with **no install on your laptop**, you:
- ran real **ns-3** (the same simulator researchers use) inside the browser
- wrote **Python** (from yesterday!) to build networks
- created **line, star, and grid** topologies and *saw* them

### Coming up
- **Tomorrow (Wed):** Jiayuan takes you deeper on the **ns-3 Web Platform** — running **C++** examples in the browser.
- The C++ you saw and the Python you wrote? ns-3 speaks **both**. You've now met each path.

💡 *Optional tonight:* change every number in this notebook at least once and find the weirdest-looking network you can make. Bring it tomorrow!